|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 1:</h2>|<h1>The Naive Loop<h1>|
|<h2>Section:</h2>|<h1>The roofline<h1>|
|<h2>Lecture:</h2>|<h1><b>Measure your own card: bandwidth, compute, and the ridge<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [1]:
# find the repo root, wherever this notebook was opened from
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))

import time
import numpy as np
import torch
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

import cudalib

# Stop using the numbers on the box

The last two notebooks used placeholder figures. Now measure your own card,
because the spec sheet and the machine disagree, and the machine is right.

In [2]:
p = torch.cuda.get_device_properties(0)
print(f'{p.name}')
print(f'  VRAM        {p.total_memory/1e9:.1f} GB')
print(f'  SMs         {p.multi_processor_count}')
print(f'  L2 cache    {p.L2_cache_size/1e6:.0f} MB')
print(f'  compute cap {p.major}.{p.minor}')

NVIDIA GeForce RTX 4080 Laptop GPU
  VRAM        12.5 GB
  SMs         58
  L2 cache    50 MB
  compute cap 8.9


### Bandwidth: copy a big block and time it

256 MB in and 256 MB out. Big enough that no cache can help, which is the
only way to measure memory rather than cache.

In [3]:
n = 256*1024*1024 // 2                       # 256 MB of bf16
a = torch.empty(n, dtype=torch.bfloat16, device='cuda')
b = torch.empty_like(a)

ms = cudalib.bench_ms(lambda: b.copy_(a), best_of=3)
bandwidth = (2 * a.numel() * 2) / (ms*1e-3)          # read + write
print(f'{ms:.3f} ms to move {4*n/1e6:.0f} MB  ->  {bandwidth/1e9:.0f} GB/s')

del a, b
torch.cuda.empty_cache()

1.669 ms to move 537 MB  ->  322 GB/s


### Compute: and why you have to measure it twice

A laptop GPU boosts to a high clock, then heats up and throttles. There is no
single number. Serving is a sustained workload, so the second one is the
honest one.

In [4]:
m = torch.randn(4096, 4096, dtype=torch.bfloat16, device='cuda')
FLOP = 2 * 4096**3

burst = FLOP / (cudalib.bench_ms(lambda: m@m, iters=30, warmup=20)*1e-3)

end = time.perf_counter() + 3.0                 # heat it up on purpose
while time.perf_counter() < end: m@m
torch.cuda.synchronize()

sustained = FLOP / (cudalib.bench_ms(lambda: m@m, iters=100, warmup=0)*1e-3)

print(f'burst     {burst/1e12:5.0f} TFLOP/s')
print(f'sustained {sustained/1e12:5.0f} TFLOP/s   ({100*sustained/burst:.0f}% of burst)')
print('\nA benchmark you run cold will lie to you, in your favour.')

burst        62 TFLOP/s
sustained    51 TFLOP/s   (82% of burst)

A benchmark you run cold will lie to you, in your favour.


# Your ridge point

In [5]:
ridge = sustained / bandwidth
print(f'  {sustained/1e12:6.1f} TFLOP/s')
print(f'  {bandwidth/1e9:6.0f} GB/s')
print(f'  ' + '-'*22)
print(f'  {ridge:6.0f} FLOP per byte\n')
print(f'At batch 1 your workload sits at 1, against {ridge:.0f}.')
print(f'That is {100/ridge:.1f}% of the arithmetic this card can do.')

    50.5 TFLOP/s
     322 GB/s
  ----------------------
     157 FLOP per byte

At batch 1 your workload sits at 1, against 157.
That is 0.6% of the arithmetic this card can do.


### What that costs, in tokens per second

At batch 1 a token costs one read of the weights. Nothing you write changes
that, so these are ceilings, not estimates.

In [6]:
print(f"{'model':<12} {'GB/step':>8} {'tok/s':>8} {'ms/token':>9}")
for name, gb in [('0.6B bf16',1.2), ('1B',2.0), ('7B',14.0), ('8B fp8',8.0)]:
  tok_s = bandwidth/1e9 / gb
  print(f'{name:<12} {gb:>8.1f} {tok_s:>8.0f} {1000/tok_s:>9.2f}')

model         GB/step    tok/s  ms/token
0.6B bf16         1.2      268      3.73
1B                2.0      161      6.22
7B               14.0       23     43.53
8B fp8            8.0       40     24.87


Beating these means raising the batch. That is the whole reason continuous
batching exists, and the next notebook shows the ceiling being hit.